# DataQualityAgent: Детектив данных

Агент для автоматического обнаружения и устранения проблем качества данных в датасете новостей.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from agents.data_quality_agent import DataQualityAgent

df = pd.read_csv('../data/raw/unified_news.csv')
print(f'Загружено {len(df)} записей')
df.head()

## Часть 1: Детектив — обнаружение проблем качества

In [ ]:
agent = DataQualityAgent()
report = agent.detect_issues(df)
print('Отчёт о качестве данных:')
print('Missing:', report['missing'])
print('Duplicates:', report['duplicates'])
print('Outliers:', {k: v.get('count_iqr', 0) for k, v in report['outliers'].items()})
print('Imbalance:', report['imbalance'])

### Визуализация: пропущенные значения

In [ ]:
if report['missing']:
    missing_df = pd.DataFrame([
        {'column': col, 'count': info['count'], 'pct': info['pct']}
        for col, info in report['missing'].items()
    ])
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=missing_df, x='column', y='count', palette='viridis', ax=ax)
    ax.set_title('Пропущенные значения по колонкам')
    ax.set_xlabel('Колонка')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### Визуализация: дубликаты

In [ ]:
dup_data = report['duplicates']
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Полные дубликаты', 'Дубликаты по URL'], [dup_data['full'], dup_data['by_url']], color=['#2ecc71', '#3498db'])
ax.set_title('Количество дубликатов')
ax.set_ylabel('Количество')
plt.tight_layout()
plt.show()

### Визуализация: выбросы (длина текста)

In [ ]:
df['text_len'] = df['text'].astype(str).str.len()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df['text_len'], vert=True)
axes[0].set_title('Boxplot: длина текста (символы)')
axes[0].set_ylabel('Символов')
axes[1].hist(df['text_len'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Распределение длины текста')
axes[1].set_xlabel('Символов')
plt.tight_layout()
plt.show()

### Визуализация: дисбаланс классов (category)

In [ ]:
if report['imbalance'] and 'category' in report['imbalance']:
    dist = report['imbalance']['category']['distribution']
    dist_series = pd.Series(dist).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(10, 5))
    dist_series.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title('Распределение категорий (дисбаланс классов)')
    ax.set_xlabel('Категория')
    ax.set_ylabel('Количество')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Часть 2: Хирург — применение стратегий чистки

In [ ]:
strategy_a = {
    'missing': 'fill_unknown',
    'duplicates': 'drop',
    'outliers': 'clip_iqr'
}
strategy_b = {
    'missing': 'drop',
    'duplicates': 'drop',
    'outliers': 'drop'
}

df_clean_a = agent.fix(df.copy(), strategy=strategy_a)
df_clean_b = agent.fix(df.copy(), strategy=strategy_b)

print(f'Стратегия A (fill_unknown + clip_iqr): {len(df)} -> {len(df_clean_a)} строк')
print(f'Стратегия B (drop): {len(df)} -> {len(df_clean_b)} строк')

### Сравнительный отчёт: стратегия A

In [ ]:
comparison_a = agent.compare(df, df_clean_a)
comparison_a

### Сравнительный отчёт: стратегия B

In [ ]:
comparison_b = agent.compare(df, df_clean_b)
comparison_b

### Сводная таблица сравнения стратегий

In [ ]:
out_a = comparison_a.loc[comparison_a['metric']=='n_outliers_text_len', 'after'].values
out_b = comparison_b.loc[comparison_b['metric']=='n_outliers_text_len', 'after'].values
summary = pd.DataFrame({
    'Метрика': ['Строк', 'Пропусков category', 'Выбросов text_len'],
    'До': [len(df), report['missing'].get('category', {}).get('count', 0), report['outliers'].get('text_len', {}).get('count_iqr', 0)],
    'Стратегия A': [len(df_clean_a), 0, out_a[0] if len(out_a) else 0],
    'Стратегия B': [len(df_clean_b), 0, out_b[0] if len(out_b) else 0],
})
summary

## Часть 3: Аргумент — обоснование выбора стратегии

**Рекомендуемая стратегия для ML-задачи (классификация по категориям, суммаризация):** стратегия A (`fill_unknown` + `clip_iqr`).

**Обоснование:**

1. **Сохранение размера выборки**: Стратегия A заполняет пропуски в `category` значением "Unknown" вместо удаления строк. Это сохраняет ~60% данных, которые иначе были бы потеряны при `drop`. Для обучения моделей суммаризации и классификации больший объём данных критичен.

2. **Обработка выбросов**: `clip_iqr` удаляет строки с экстремальной длиной текста (очень короткие или очень длинные статьи). Это устраняет шум и аномалии, которые могут исказить метрики и качество моделей. Альтернатива `drop` для outliers даёт аналогичный эффект.

3. **Дисбаланс классов**: При `fill_unknown` появляется дополнительный класс "Unknown" — это явно маркирует неразмеченные данные. При `drop` мы теряем все такие записи, что может сместить распределение в сторону источников с разметкой (например, Gazeta).

4. **Компромисс**: Стратегия B (drop) даёт «чище» данные без Unknown, но сильно сокращает датасет. Для production-моделей с ограниченными данными стратегия A предпочтительнее.

### Бонус: LLM-скилл (explain_and_recommend)

In [ ]:
# Раскомментировать при наличии ANTHROPIC_API_KEY
# rec = agent.explain_and_recommend(report, 'Классификация новостей по категориям, суммаризация текстов')
# print(rec)